# Multi-Class News Classification using PyTorch & TorchText
**Author:** Ibrahim Hamada Mosaad  
**Role:** AI Engineer & Researcher

###  Introduction
This notebook explores the task of classifying news articles into four categories: **World, Sports, Business, and Sci/Tech**. We leverage the `AG_NEWS` dataset and implement a neural network using `nn.EmbeddingBag` for efficient text representation.

In [1]:
# 1. Setup and Library Installation
try:
    import torch
    import torchtext
except ImportError:
    %pip install torch==2.3.0 torchtext==0.18.0 torchdata==0.9.0 --index-url https://download.pytorch.org/whl/cpu
    %pip install pandas numpy==1.26.4 matplotlib scikit-learn plotly tqdm portalocker>=2.0.0

In [2]:
# 2. Imports
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchtext.datasets import AG_NEWS
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.functional import to_map_style_dataset
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


###  Data Exploration & Preprocessing
We use the `AG_NEWS` dataset. First, we need to tokenize the text and build a vocabulary[cite: 1].

In [3]:
tokenizer = get_tokenizer("basic_english")
train_iter = AG_NEWS(split='train')

def yield_tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text.lower())

# Build Vocabulary
vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

text_pipeline = lambda x: vocab(tokenizer(x))
label_pipeline = lambda x: int(x) - 1

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 95811


###  Model Architecture
The `EmbeddingBag` layer is used here to calculate the average of word embeddings in a document, which is memory-efficient and fast[cite: 1].

In [4]:
class TextClassificationModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super(TextClassificationModel, self).__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False)
        self.fc = nn.Linear(embed_dim, num_class)
        self.init_weights()

    def init_weights(self):
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        return self.fc(embedded)

num_class = 4
emsize = 64
model = TextClassificationModel(len(vocab), emsize, num_class).to(device)

###  Training Phase
We use Stochastic Gradient Descent (SGD) and a Step Learning Rate scheduler[cite: 1].

In [5]:
def collate_batch(batch):
    label_list, text_list, offsets = [], [], [0]
    for _label, _text in batch:
        label_list.append(label_pipeline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
        offsets.append(processed_text.size(0))
    label_list = torch.tensor(label_list, dtype=torch.int64)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    text_list = torch.cat(text_list)
    return label_list.to(device), text_list.to(device), offsets.to(device)

# Data Loaders
train_iter, test_iter = AG_NEWS()
train_dataset = to_map_style_dataset(train_iter)
num_train = int(len(train_dataset) * 0.95)
split_train_, split_valid_ = random_split(train_dataset, [num_train, len(train_dataset) - num_train])

train_dataloader = DataLoader(split_train_, batch_size=64, shuffle=True, collate_fn=collate_batch)

# Hyperparameters
EPOCHS = 5 # Reduced for quick exploration
LR = 0.1
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.1)

print("Starting Training...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    for idx, (label, text, offsets) in enumerate(tqdm(train_dataloader)):
        optimizer.zero_grad()
        predicted_label = model(text, offsets)
        loss = criterion(predicted_label, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
    print(f"End of Epoch {epoch}")

Starting Training...


100%|██████████| 1782/1782 [00:29<00:00, 60.04it/s]


End of Epoch 1


100%|██████████| 1782/1782 [00:30<00:00, 57.50it/s]


End of Epoch 2


...

End of Epoch 3


100%|██████████| 1782/1782 [00:28<00:00, 62.03it/s]


End of Epoch 4


100%|██████████| 1782/1782 [00:28<00:00, 62.10it/s]

End of Epoch 5


###  Inference
Test the model with a custom news headline[cite: 1].

In [6]:
ag_news_label = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tec"}

def predict(text):
    with torch.no_grad():
        text_tensor = torch.tensor(text_pipeline(text))
        output = model(text_tensor, torch.tensor([0]))
        return ag_news_label[output.argmax(1).item() + 1]

ex_text = "The stock market saw a significant rise following the new fiscal policy."
print(f"Category: {predict(ex_text)}")

Category: Business
